In [ ]:
import fiftyone as fo

dataset = fo.Dataset.from_dir(
    dataset_dir="/Data_large/marine/Datasets/V2RAW/B_02_03_04_08_update/imgs",
    dataset_type=fo.types.COCODetectionDataset,
    labels_path="/Data_large/marine/Datasets/V2RAW/B_02_03_04_08_update/imgs/data/val.json"
)

In [ ]:
session = fo.launch_app(dataset)

# Requirements:

In [ ]:
import tifffile
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
from pathlib import Path 

def process_tif_image(tif_image, threshold):
    """
    Process a 16-bit TIFF image by normalizing it to 8-bit and applying histogram equalization.

    Parameters:
    tif_image (numpy.ndarray): A 16-bit TIFF image with multiple channels.

    Returns:
    numpy.ndarray: A processed image where bands b2, b3, and b4 are merged after normalization and histogram equalization.
    """

    # Check the shape of the image to understand the number of channels
    print(f"Image shape: {tif_image.shape}")

    # Step 1: Read the 16-bit image bands
    b2, b3, b4, b8 = tif_image[:, :, 0], tif_image[:, :, 1], tif_image[:, :, 2], tif_image[:, :, 3]

    # Step 2: Normalize the image to 8-bit
    # Assuming the image uses the full 16-bit range (0-65535)
    norm_bands = []
    for band in [b2, b3, b4, b8]:
        norm_band = (255 * (band / threshold)).astype(np.uint8)
        norm_bands.append(norm_band)

    # Step 3: Apply histogram equalization
    equ_bands = []
    for band in norm_bands:
        eq = cv2.equalizeHist(band)
        equ_bands.append(eq)

    # Step 4: Merge b2, b3, and b4 bands
    merged_image = cv2.merge((equ_bands[0], equ_bands[1], equ_bands[2]))

    return merged_image


def save_image_as_jpg(image, filename):
    """
    Save an image as a JPG file.

    Parameters:
    image (numpy.ndarray): The image to be saved.
    filename (str): The name of the file to save the image as. It should include the .jpg extension.

    Returns:
    bool: True if the image is successfully saved, False otherwise.
    """
    success = cv2.imwrite(filename, image)
    return success

# Example usage
# tif_path = '/Data_large/marine/Datasets/V2RAW/B_02_03_04_08_update/imgs/imgs_B2B3B4B8/day1_g_0_coreg.tif'

tif_folder = '/Data_large/marine/Datasets/V2RAW/B_02_03_04_08_update/imgs/imgs_B2B3B4B8'
tif_paths = list(Path(tif_folder).iterdir())
for tif_path in tif_paths:
    tif_image = tifffile.imread(tif_path)
    processed_image = process_tif_image(tif_image, threshold=100)
    save_image_as_jpg(processed_image, filename=f'/Data_large/marine/Datasets/V2RAW/B_02_03_04_08_update/imgs/imgs_B3B3B4B8_jpg/{tif_path.stem}.jpg')


In [ ]:
# Read the 16-bit multi-channel TIFF image
tif_path = '/Data_large/marine/Datasets/V2RAW/B_02_03_04_08_update/imgs/imgs_B2B3B4B8/day1_g_0_coreg.tif'
tif_image = tifffile.imread(tif_path)

# Check the shape of the image to understand the number of channels
print(f"Image shape: {tif_image.shape}")
# Step 1: Read the 16-bit image
b2,b3,b4,b8 = tif_image[:,:,0], tif_image[:,:,1], tif_image[:,:,2], tif_image[:,:,3]
# Step 2: Normalize the image to 8-bit
# Assuming the image uses the full 16-bit range (0-65535)

norm_bands = []
for band in [b2,b3,b4,b8]:
    norm_band = (255 * (band / 50.0)).astype(np.uint8)
    norm_bands.append(norm_band)


# Step 3: Apply histogram equalization
equ_bands = []
for band in norm_bands:
    eq = cv2.equalizeHist(band)
    equ_bands.append(eq)

# plt.figure(dpi=220)
# plt.imshow([equ_bands[:3]])
# plt.show()

# # Save or display the result
# cv2.imwrite('equalized_image.tif', equalized_image)
# cv2.imshow('Equalized Image', equalized_image)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

# print(f"The image has been successfully converted and saved as {jpeg_path}")

In [ ]:
for b in equ_bands:
    print(b.mean(), b.std())
    plt.figure(dpi=120)
    plt.imshow(b)
    plt.show()